<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 1</center></font>

<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>LLM-Powered Medical Assistant</center></font>

## Problem Statement

### Business Context

**MedCore Health Network** is a mid-sized multi-specialty hospital chain operating 12 facilities across urban and semi-urban India, handling 3,000+ patient consultations daily across emergency, internal medicine, and diagnostics departments.

Physicians and residents currently cross-check diagnoses using printed reference manuals, fragmented internal wikis, and ad-hoc web searches. There is no centralized, searchable link between clinical queries and trusted medical literature such as *Harrison's Principles of Internal Medicine*, *Merck Manual*, and WHO guidelines. A complex case typically requires **8–12 minutes** of reference lookup, contributing to delays in time-sensitive settings such as the Emergency Room.

Physicians are time-constrained and require trustworthy, explainable AI outputs; clinical reference lookup is currently informal, with no SOP or audit trail for source verification; and approved medical manuals remain in static PDF and print formats without a searchable index, API, or structured knowledge base.

MedCore previously piloted a basic keyword-search intranet tool using digitized manual excerpts. The pilot was discontinued after four months because it relied on exact-term matching. Clinically phrased queries such as *"chest pain radiating to left arm, elderly diabetic"* often returned poor or no results when the wording differed from the terminology in the source material.

The pilot showed that physicians were willing to use a digital reference tool, but the system failed to understand clinical language. The gap was therefore primarily **platform and process related**, rather than a physician adoption problem.

The current initiative uses **retrieval-augmented generation (RAG)** to address this limitation through semantic retrieval while keeping responses traceable to approved source material.

**Common Questions to Answer**

1. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

2. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

3. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

### Objective

Design and build a **RAG-based clinical decision-support prototype** for the **Internal Medicine department** that ingests renowned medical manuals and provides evidence-grounded answers to physician queries in **under 30 seconds**, with the business goal of reducing reference-lookup time from the current **8–12 minutes to under 2 minutes**.

The prototype should retrieve relevant evidence from approved manuals and generate responses that are constrained to the retrieved source material, with citations surfaced so physicians can verify the underlying evidence.

The system is intended as a **clinical reference and lookup accelerator**, not a diagnostic authority, and does not replace physician judgment.

Every generated response should be evaluated before being presented as a demo result.

**Evaluation Criteria**

- **Relevance:** Measures whether retrieved context matches the intent of the physician query. Target: **≥0.85** across 50 physician-authored test queries. This is the leading metric and is evaluated before answer generation.
- **Groundedness:** Measures whether the generated answer is fully supported by the retrieved source material. Target: **≥0.90**, with unsupported clinical claims rejected. This is the lagging metric and is evaluated after generation.


**Key Risks & Mitigations**

| **Risk** | **Type** | **Mitigation** |
|---|---|---|
| Hallucinated clinical claims not present in source manuals | Technology | Enforce citation-constrained generation and reject ungrounded outputs during evaluation |
| Physician distrust of AI-generated answers | People | Surface source citations with every answer and position the system as a lookup accelerator rather than a diagnostic authority |
| Low physician adoption, repeating the 2023 pilot outcome | Process | Pilot with a small physician cohort and collect qualitative feedback before wider rollout |

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Solution Approach

* **LLM-Only Answer**

  * Start by asking the LLM to answer the medical questions using its general knowledge.

* **Build Medical Knowledge Base**

  * Prepare the medical manual so it can be searched efficiently.
  * Convert the medical content into embeddings and store it in Chroma.
  * Set up a retriever to find the most relevant medical information for each question.

* **RAG Answer**

  * Retrieve relevant information from the medical knowledge base for each question.
  * Provide the retrieved context to the LLM to generate a grounded medical answer.

* **Evaluate Answers**

  * Check whether the generated answers are supported by the retrieved medical context.
  * Check whether the answers directly address the questions.

* **Tune RAG**

  * Experiment with different retrieval and generation settings.
  * Compare the tuned RAG responses with the baseline to understand the impact of these parameters.


## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Install required libraries
!pip install -q langchain_community \
              langchain \
              chromadb \
              pymupdf \
              tiktoken \
              datasets \
              evaluate \
              langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/13

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# Core Python libraries
import os
import json
import re
from pydantic import BaseModel, Field

# Data handling and tokenization
import pandas as pd
import tiktoken

# OpenAI API
from openai import OpenAI

# Dataset handling
from datasets import Dataset

# PDF loading
from langchain_community.document_loaders import PyMuPDFLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# OpenAI embeddings
from langchain_openai.embeddings import OpenAIEmbeddings

# Vector database
from langchain_community.vectorstores import Chroma

# OpenAI chat model
from langchain_openai import ChatOpenAI

/tmp/ipykernel_2180/2751444389.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


### OpenAI API Calling



> Note: Please upload the **config file** before running the cell below. Refer to the **instruction page** for steps to store your API key in the config file and load it into the notebook.


In [2]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

## Question Answering using LLM with Prompt Engineering

In this section, we will:

1. **Define the system prompt:** Create one system prompt to guide the model's responses.
2. **Create a reusable function:** Build a function that takes the system prompt and user question as inputs and returns the model's response.
3. **Generate responses:** Use the reusable function to generate answers for all three questions.


**System Prompt**

A well-structured prompt typically includes:

- **Role:** Defines who the model should act as.
- **Task:** Defines what the model needs to do.
- **Guidelines:** Defines how the model should perform the task and the expected response behavior.

In [3]:
system_prompt = """
### ROLE
You are an AI assistant specializing in medical knowledge. Your role is to provide clear, precise, and medically reliable responses based on established medical guidelines and best practices.

### TASK
When answering, prioritize factual correctness, align with widely accepted medical standards, and ensure clarity for both medical professionals and general users.
If a query requires specific reference materials beyond general medical knowledge, acknowledge the limitation rather than speculating.

### GUIDELINES
- Prioritize factual accuracy and clarity.
- Avoid speculation or unsupported claims.
- Clearly acknowledge limitations when information is uncertain or unavailable.
"""

***Prompt***:

<font size=3 color="#4682B4"><b> Create a function to query the OpenAI model using user input and return the generated response.
</font>

This function takes a system prompt and user question, sends them to the OpenAI chat model, and returns the generated response.


**Model Parameters**

- **Max Tokens:** Controls the maximum length of the response.
- **Temperature (0–1):** Controls randomness. **0** = more deterministic and focused; **1** = more random and creative.
- **Top-p (0–1):** Controls token diversity. **0** = considers fewer likely tokens; **1** = considers a broader range of likely tokens.

> **Note:** We keep `temperature = 0` throughout this notebook to generate focused and consistent responses, which is important for medical use.


In [4]:
GENERATION_MODEL = "gpt-4o-mini"

In [5]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model=GENERATION_MODEL,                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content.strip()                                # Return the text content from the model's reply

In [6]:
questions = {
    "Question_1": "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "Question_2": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "Question_3": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
}

**Question1: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [7]:
response_with_prompt_eng_1 = response(system_prompt, questions['Question_1'])
print(response_with_prompt_eng_1)

Common symptoms of appendicitis include:

1. **Abdominal Pain**: Typically starts around the navel and then shifts to the lower right abdomen.
2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.
3. **Loss of Appetite**: A common symptom that accompanies the pain.
4. **Fever**: Mild fever may develop as the condition progresses.
5. **Constipation or Diarrhea**: Some patients may experience changes in bowel habits.
6. **Abdominal Swelling**: In some cases, the abdomen may become swollen.

Appendicitis is primarily treated through surgical intervention, as it cannot be effectively cured with medication alone. The standard surgical procedure for appendicitis is an **appendectomy**, which involves the removal of the inflamed appendix. This can be performed using two main techniques:

1. **Open Appendectomy**: A larger incision is made in the lower right abdomen to remove the appendix.
2. **Laparoscopic Appendectomy**: A minimally invasive technique using small incisions a

**Question 2: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [8]:
response_with_prompt_eng_2 = response(system_prompt, questions["Question_2"])
print(response_with_prompt_eng_2)

Sudden patchy hair loss, often referred to as alopecia areata, can manifest as localized bald spots on the scalp or other areas of the body. Here’s an overview of potential causes and effective treatments:

### Possible Causes
1. **Alopecia Areata**: An autoimmune condition where the immune system attacks hair follicles, leading to hair loss.
2. **Stress**: Physical or emotional stress can trigger hair loss in some individuals.
3. **Genetics**: A family history of hair loss can increase susceptibility.
4. **Hormonal Changes**: Conditions such as thyroid disorders or hormonal imbalances can contribute to hair loss.
5. **Nutritional Deficiencies**: Lack of essential nutrients, such as iron, zinc, or vitamins, can affect hair health.
6. **Infections**: Fungal infections like tinea capitis can cause patchy hair loss.
7. **Other Medical Conditions**: Conditions like lupus or vitiligo may also lead to hair loss.

### Effective Treatments
1. **Topical Corticosteroids**: These are often the fi

**Question 3:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [9]:
response_with_prompt_eng_3 = response(system_prompt, questions["Question_3"])
print(response_with_prompt_eng_3)

The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), depends on the severity of the injury and the specific symptoms presented. Here are some general approaches to treatment:

1. **Immediate Medical Care**:
   - **Emergency Treatment**: In cases of severe injury, immediate medical attention is critical. This may involve stabilizing the patient, ensuring adequate oxygenation, and managing intracranial pressure.
   - **Surgery**: Surgical intervention may be necessary to remove hematomas, repair skull fractures, or relieve pressure on the brain.

2. **Acute Care**:
   - **Monitoring**: Patients are often monitored in a hospital setting for neurological status, vital signs, and potential complications.
   - **Medications**: Medications may be administered to manage symptoms such as pain, seizures, or swelling (e.g., corticosteroids).

3. **Rehabilitation**:
   - **Physical Therapy**: To improve mobility and strength.
   - 

**Observation**

* All three questions are answered using the same prompt-engineering approach, and the responses appear relevant and well-structured.


#### Store the Results

Now, we store the generated answers in a DataFrame so that we can evaluate and compare them later.


In [10]:
# Create the DataFrame
result_df = pd.DataFrame({
    "questions": [questions["Question_1"], questions["Question_2"], questions["Question_3"]],
    "prompt_eng_response": [response_with_prompt_eng_1, response_with_prompt_eng_2, response_with_prompt_eng_3]})

# Display the DataFrame
result_df.head()

,questions,prompt_eng_response
0,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....
1,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ..."
2,What treatments are recommended for a person w...,The treatment for a person who has sustained a...


## Data Preparation for RAG

### Loading the Data

In [11]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [12]:
# Set the path to the PDF file
manual_pdf_path = "/content/medical_diagnosis_manual.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()                                                      # Load and extract text from all pages of the PDF

### Data Chunking

**Text Chunking**

We define a text splitter to divide the document into smaller, overlapping chunks using OpenAI's `cl100k_base` tokenizer.

- **Chunk size:** 512 tokens
- **Chunk overlap:** 50 tokens

In [13]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=512,                                                             # Each chunk will have up to 512 tokens
    chunk_overlap=50                                                            # 50 tokens will overlap between consecutive chunks (for context continuity)
)

Use the text splitter to divide the PDF content into smaller, manageable chunks that can be processed and retrieved efficiently.

In [14]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)                      # Returns a list of chunked documents

In [15]:
len(document_chunks)                                                            # Total number of text chunks generated from the PDF

8308

**Observation:** 8,308 chunks were created from the PDF document.

### Load Embedding Model

Define the OpenAI embedding model and generate a vector representation for the first document chunk.

In [16]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

Dimension of the embedding vector  1536


**Observation:** The embedding vector has a dimension of **1,536**, meaning each document chunk is represented as a 1,536-dimensional numerical vector.

In [17]:
# Return/display the two embedding vectors for further inspection or use
embedding_1[:10]

[0.017092563211917877,
 0.012418919242918491,
 0.034820638597011566,
 -0.017423558980226517,
 -0.019409526139497757,
 0.008804457262158394,
 -0.00787105318158865,
 -0.03500599414110184,
 -0.007559917867183685,
 -0.01099564228206873]

**Observation:** The first 10 values of the embedding vector are displayed, representing a small portion of the 1,536-dimensional vector generated for the document chunk.

### Vector Database Creation

We check whether a persistent vector store already exists. If it exists, we load it; otherwise, we create a new vector store from the document chunks and save it for future use.

In [18]:
# Create or load the persistent vector store
out_dir = "medical_db"

if os.path.exists(out_dir) and os.listdir(out_dir):
    # Load the existing vector store
    vectorstore = Chroma(
        persist_directory=out_dir,
        embedding_function=embedding_model
    )
    print("Existing vector store loaded successfully.")

else:
    # Create the vector store and persist it to disk
    vectorstore = Chroma.from_documents(
        document_chunks,
        embedding_model,
        persist_directory=out_dir
    )
    print("New vector store created and saved successfully.")

New vector store created and saved successfully.


## Question Answering using RAG

**System and User Prompt Template**

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

The structure of this prompt is largely the same as the previous prompt, with two additional components:

- **Context:** Defines the information provided to the model for answering the query.
- **Output Format:** Defines how the model should structure the final response.

In [19]:
# Define the system prompt for the model
qna_system_message = """
### ROLE
You are an AI assistant designed to support professional doctors. Your task is to provide evidence-based, concise, and relevant medical information to doctors' clinical questions based on the context provided.

### CONTEXT
User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of trusted medical literature and research articles relevant to the query, along with their source details.

### TASK
When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable medical insights.
3. Include the source reference with the page number, journal name, or publication, as provided.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

### GUIDELINES
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

### OUTPUT FORMAT
Here is an example of how to structure your response:

Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

In [20]:
# Define the user message template
qna_user_message_template = """

###Context
Here are some excerpts from medical literature and their sources that are relevant to the clinical question mentioned below:

{context}

###Question

{question}
"""

**Response Function**

***Prompt***:

<font size=3 color="#4682B4"><b> Write a Python `generate_rag_response()` function that retrieves the top `k` chunks from `vectorstore`, adds source/page citations, builds the context, inserts it into `qna_user_message_template`, and generates a response using `gpt-4o-mini` with `qna_system_message`, `max_tokens`, `temperature`, and `top_p`. Include error handling and return both the response and retrieved context. Use the existing `client` and variables.

</font>

This function
1. retrieves the most relevant document chunks for a user query,
2. adds them to the prompt as context, and
3. uses the LLM to generate a grounded response.


In [44]:
def generate_rag_response(user_input, k=3, max_tokens=1000, temperature=0, top_p=0.95):

    global qna_system_message, qna_user_message_template

    ### 1. Retrieve relevant chunks

    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    relevant_document_chunks = retriever.invoke(user_input)

    context_list = []
    for d in relevant_document_chunks:
        source = d.metadata.get("source", "Unknown source")
        page = d.metadata.get("page", "Unknown page")
        page_display = page + 1 if isinstance(page, int) else page
        chunk_with_citation = f"[Source: {source} | Page: {page_display}]\n{d.page_content}"
        context_list.append(chunk_with_citation)

    context_for_query = "\n\n---\n\n".join(context_list)

    ### 2. Add the context in the prompt
    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    ### 3. Generate response using LLM
    try:
        llm_response_content = globals()['response'](qna_system_message, user_message, max_tokens, temperature, top_p)

    except Exception as e:
        llm_response_content = f'Sorry, I encountered the following error: \n {e}'

    return llm_response_content, context_for_query

**Question1: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [45]:
response_with_rag_1, context_1 = generate_rag_response(questions["Question_1"])
print(response_with_rag_1)

Answer:
The common symptoms of appendicitis include epigastric or periumbilical pain followed by nausea, vomiting, and anorexia, with pain typically shifting to the right lower quadrant after a few hours. Classic signs include right lower quadrant tenderness at McBurney's point, Rovsing sign, psoas sign, and obturator sign. Low-grade fever is also common. 

Appendicitis cannot be cured with medicine alone; the treatment is surgical removal of the appendix, either through open or laparoscopic appendectomy. Preceding the surgery, IV antibiotics are recommended, especially if the appendix is perforated.

Source:
/content/medical_diagnosis_manual.pdf | Page: 174-175


**Question 2: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [46]:
response_with_rag_2, context_2 = generate_rag_response(questions["Question_2"])
print(response_with_rag_2)

Answer:
Sudden patchy hair loss, known as alopecia areata, is thought to be an autoimmune disorder affecting genetically susceptible individuals. Effective treatments include topical corticosteroids, oral antimalarials, and immunosuppressants. Microscopic hair examination or scalp biopsy may be required for definitive diagnosis, especially if the hair loss persists.

Possible causes of sudden patchy hair loss include autoimmune disorders, stress, and environmental triggers, although the exact triggers remain unclear.

Source:
The Merck Manual of Diagnosis & Therapy, 19th Edition | Pages: 849, 857, 858


**Question 3:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [47]:
response_with_rag_3, context_3 = generate_rag_response(questions["Question_3"])
print(response_with_rag_3)

Answer:
Initial treatment for traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be necessary for severe injuries to monitor and treat intracranial pressure, decompress the brain, or remove hematomas. In the days following the injury, it is crucial to maintain adequate brain perfusion and oxygenation while preventing complications. Subsequently, many patients require rehabilitation, which should be planned early and may involve a team approach that includes physical, occupational, and speech therapy, as well as counseling to address social and emotional needs.

Source:
The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 324. Traumatic Brain Injury, Pages 3405-3413.


**Observation:**

- The BASE RAG response is more concise and focused, providing shorter answers based on the retrieved context.
- Each answer is supported by and marked with the relevant source, making it easier for physicians to verify the information.

#### Store the Results

We store the generated answers in a DataFrame so they can be evaluated and compared in the next section.

In [48]:
# Add the results to a new column in the DataFrame
result_df['responses_with_RAG'] = [response_with_rag_1, response_with_rag_2, response_with_rag_3]
result_df["rag_context"] = [context_1, context_2, context_3]
# Display the DataFrame
result_df.head()

,questions,prompt_eng_response,responses_with_RAG,rag_context,prompt_eng_groundedness,prompt_eng_groundedness_rationale,prompt_eng_relevance,prompt_eng_relevance_rationale,rag_groundedness,rag_groundedness_rationale,rag_relevance,rag_relevance_rationale
0,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....,Answer:\nThe common symptoms of appendicitis i...,[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer is fully grounded in the provided c...,1.0,The answer directly addresses the user's quest...,0.0,The answer provided does not contain any subst...,0.0,The answer does not address the question at al...
1,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ...","Answer:\nSudden patchy hair loss, known as alo...",[Source: /content/medical_diagnosis_manual.pdf...,0.6,The answer provides a comprehensive overview o...,1.0,The answer is fully relevant as it directly ad...,0.0,The answer provided does not contain any subst...,0.0,The answer does not address the question at al...
2,What treatments are recommended for a person w...,The treatment for a person who has sustained a...,Answer:\nInitial treatment for traumatic brain...,[Source: /content/medical_diagnosis_manual.pdf...,0.8,The answer is mostly grounded in the context p...,1.0,The answer is fully relevant as it directly ad...,0.0,The answer provided does not contain any subst...,0.0,The answer does not address the question at al...


## Evaluation


#### Defining required System Prompts

In the evaluation prompts, we add these key components to make the evaluation structured, consistent, and reliable.

- **Evaluation Criteria:** Defines the specific aspects of the answer that the evaluator should assess.
- **Scoring Rubric:** Defines what each score represents, helping the evaluator assign scores consistently.
- **Evaluation Process:** Provides clear steps for analyzing the answer and arriving at the final score.
- **Score + Rationale:** The score provides a measurable evaluation, while the rationale explains why that score was assigned and makes the evaluation easier to interpret.

In [26]:
groundedness_rater_system_message = """
### ROLE
You are an evaluator assessing the quality of an AI-generated answer to a user's question.

### INPUT CONTEXT
You will be provided with:
- ###Question: the user's question
- ###Context: the information available to support the answer
- ###Answer: the AI-generated answer

### TASK
Your task is to evaluate the answer for GROUNDEDNESS.

### METRIC
Groundedness measures the extent to which the claims made in the answer are supported by the information provided in the context.

### EVALUATION CRITERIA
Evaluation should consider:
- Whether the factual claims in the answer can be supported by the context.
- Whether the answer introduces information, facts, conclusions, or details that are not supported by the context.
- Whether the answer accurately represents the information available in the context.
- Whether the answer avoids unsupported assumptions or fabricated information.

### IMPORTANT
The answer should be evaluated only against the provided context. Do not use outside knowledge to determine whether a claim is true.

### SCORING RUBRIC
- 1.0: Fully grounded. All substantive claims in the answer are directly supported by the context, with no unsupported claims.
- 0.8: Mostly grounded. Nearly all substantive claims are supported, with only minor unsupported details that do not materially affect the answer.
- 0.6: Moderately grounded. Most of the answer is supported, but there are some noticeable unsupported claims or details.
- 0.4: Partially grounded. A significant portion of the answer is unsupported by the context.
- 0.2: Minimally grounded. Most substantive claims are unsupported by the context.
- 0.0: Not grounded. The answer is unsupported by the context or substantially contradicts the information provided.

The score may be any value between 0.0 and 1.0, including intermediate values such as 0.1, 0.3, 0.5, 0.7, or 0.9. Choose the score that best reflects the overall degree of groundedness.

### EVALUATION PROCESS
1. Identify the factual and substantive claims made in the answer.
2. Locate supporting information for each claim in the provided context.
3. Identify claims that are unsupported, assumed, or inconsistent with the context.
4. Assess how much of the answer is supported by the context.
5. Determine the overall degree of groundedness based on the scoring rubric.
6. Assign a score between 0.0 and 1.0.
7. Provide a brief rationale explaining why the assigned score is appropriate.

### OUTPUT FORMAT
Return the groundedness score and a brief rationale in dictionary format.

Example:
{groundedness_score:0.7, rationale:"Most claims are supported by the context, but one key detail is not explicitly supported."}
"""

In [27]:
relevance_rater_system_message = """
### ROLE
You are an evaluator assessing the quality of an AI-generated answer to a user's question.

### INPUT CONTEXT
You will be provided with:
- ###Question: the user's question
- ###Answer: the AI-generated answer

### TASK
Your task is to evaluate the answer for RELEVANCE.

### METRIC
Relevance measures how well the answer directly addresses the user's question.

### EVALUATION CRITERIA
Evaluation should consider:
- Whether the answer directly addresses the user's question.
- Whether the answer covers the important aspects of the question.
- Whether the answer is focused and useful.
- Whether the answer avoids unnecessary or unrelated information.

### IMPORTANT
Do not evaluate whether the answer is factually correct or supported by a reference context.
Evaluate only how well the answer addresses the question.

### SCORING RUBRIC
- 1.0: Fully relevant. The answer directly addresses all important aspects of the question with no meaningful irrelevant content.
- 0.8: Mostly relevant. The answer addresses nearly all important aspects, with only minor omissions or irrelevant details.
- 0.6: Moderately relevant. The answer addresses the main question but misses some important aspects or includes some unnecessary information.
- 0.4: Partially relevant. The answer addresses only some aspects of the question and has noticeable omissions or irrelevant content.
- 0.2: Minimally relevant. The answer has limited connection to the question and fails to address most important aspects.
- 0.0: Not relevant. The answer does not meaningfully address the question.

The score may be any value between 0.0 and 1.0, including intermediate values such as 0.1, 0.3, 0.5, 0.7, or 0.9. Choose the score that best reflects the overall degree of relevance.

### EVALUATION PROCESS
1. Identify the main information requirements in the question.
2. Check whether the answer addresses those requirements.
3. Check whether the answer contains unnecessary or unrelated information.
4. Determine the overall degree of relevance.
5. Assign a score between 0.0 and 1.0.
6. Provide a brief rationale explaining why the assigned score is appropriate.

### OUTPUT FORMAT
Return the relevance score and a brief rationale in dictionary format.

Example:
{relevance_score:0.7, rationale:"The answer addresses the main question but misses one important aspect."}
"""

In [28]:
groundedness_user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [29]:
relevance_user_message_template = """
###Question
{question}

###Answer
{answer}
"""

#### Utility Function

The `EvaluationResult` schema defines the structured format returned by the evaluation model.

- **`score`**: A numerical score between **0 and 1**, where higher values indicate better evaluation performance.
- **`rationale`**: A brief explanation from the evaluation model describing the reasoning behind the assigned score.

This structured output makes the evaluation results consistent and easier to analyze across responses.

In [30]:
class EvaluationResult(BaseModel):
    score: float = Field(ge=0.0, le=1.0)
    rationale: str

This function uses an evaluation model to assess a response against a given prompt and returns the result in a structured **EvaluationResult** format.


> **Note:** We use **GPT-4o** as the evaluation model, a higher-capability model than the **GPT-4o-mini** used for response generation.


In [31]:
JUDGE_MODEL = "gpt-4o"

In [32]:
def evaluate_response(system_message, user_message):
    completion = client.beta.chat.completions.parse(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        response_format=EvaluationResult,
        temperature=0,
        top_p=1,
        max_tokens=1000
    )

    return completion.choices[0].message.parsed

***Prompt***:

<font size=3 color="#4682B4"><b> Write a Python function `evaluate_answers` that iterates through questions, generated answers, and retrieved contexts, evaluates each answer for groundedness and relevance using `evaluate_response`, and returns both evaluation results.

</font>

This function evaluates each generated answer for **groundedness** against the retrieved context and **relevance** to the question using an evaluation model.


In [33]:
def evaluate_answers(questions, answers, contexts):

    groundedness_results = []
    relevance_results = []

    for question, answer, context in zip(questions, answers, contexts):

        groundedness_result = evaluate_response(
            groundedness_rater_system_message,
            groundedness_user_message_template.format(
                context=context,
                question=question,
                answer=answer
            )
        )

        relevance_result = evaluate_response(
            relevance_rater_system_message,
            relevance_user_message_template.format(
                question=question,
                answer=answer
            )
        )

        groundedness_results.append(groundedness_result)
        relevance_results.append(relevance_result)

    return groundedness_results, relevance_results

#### Evaluation 1: Prompt Engineering Response Evaluation

Evaluate the prompt-engineered responses and store their groundedness and relevance scores for comparison.

**Note:** Prompt-engineered responses are evaluated against the same context retrieved for RAG to ensure the answer remains grounded in the available evidence.


In [34]:
prompt_eng_questions = list(result_df["questions"])
prompt_eng_answers = list(result_df["prompt_eng_response"])
baseline_contexts = list(result_df["rag_context"])

prompt_eng_groundedness, prompt_eng_relevance = evaluate_answers(
    prompt_eng_questions,
    prompt_eng_answers,
    baseline_contexts
)

In [35]:
# Store scores and rationales
result_df["prompt_eng_groundedness"] = [result.score for result in prompt_eng_groundedness]

result_df["prompt_eng_groundedness_rationale"] = [result.rationale for result in prompt_eng_groundedness]

result_df["prompt_eng_relevance"] = [result.score for result in prompt_eng_relevance]

result_df["prompt_eng_relevance_rationale"] = [result.rationale for result in prompt_eng_relevance]

result_df[
    [
        "questions",
        "prompt_eng_groundedness",
        "prompt_eng_groundedness_rationale",
        "prompt_eng_relevance",
        "prompt_eng_relevance_rationale"
    ]
]

,questions,prompt_eng_groundedness,prompt_eng_groundedness_rationale,prompt_eng_relevance,prompt_eng_relevance_rationale
0,"What are the common symptoms for appendicitis,...",1.0,The answer is fully grounded in the provided c...,1.0,The answer directly addresses the user's quest...
1,What are the effective treatments or solutions...,0.6,The answer provides a comprehensive overview o...,1.0,The answer is fully relevant as it directly ad...
2,What treatments are recommended for a person w...,0.8,The answer is mostly grounded in the context p...,1.0,The answer is fully relevant as it directly ad...


**Observation:**

- Prompt engineering achieves consistently high relevance (1.0), but varying groundedness (0.6–0.9) shows that relevant answers may still include claims not fully supported by the available context.


#### Evaluation 2: RAG Response Evaluation

In [49]:
baseline_questions = list(result_df["questions"])
baseline_answers = list(result_df["responses_with_RAG"])
baseline_contexts = list(result_df["rag_context"])

baseline_groundedness, baseline_relevance = evaluate_answers(
    baseline_questions,
    baseline_answers,
    baseline_contexts
)

In [50]:
# Store scores and rationales
result_df["rag_groundedness"] = [result.score for result in baseline_groundedness]

result_df["rag_groundedness_rationale"] = [result.rationale for result in baseline_groundedness]

result_df["rag_relevance"] = [result.score for result in baseline_relevance]

result_df["rag_relevance_rationale"] = [result.rationale for result in baseline_relevance]

result_df[
    [
        "questions",
        "responses_with_RAG",
        "rag_groundedness",
        "rag_groundedness_rationale",
        "rag_relevance",
        "rag_relevance_rationale"
    ]
]

,questions,responses_with_RAG,rag_groundedness,rag_groundedness_rationale,rag_relevance,rag_relevance_rationale
0,"What are the common symptoms for appendicitis,...",Answer:\nThe common symptoms of appendicitis i...,1.0,The answer accurately reflects the information...,1.0,The answer directly addresses the user's quest...
1,What are the effective treatments or solutions...,"Answer:\nSudden patchy hair loss, known as alo...",0.9,The answer is mostly grounded in the context p...,1.0,The answer directly addresses the user's quest...
2,What treatments are recommended for a person w...,Answer:\nInitial treatment for traumatic brain...,1.0,The answer is fully grounded in the provided c...,1.0,The answer directly addresses the question by ...


**Observation:**
- RAG achieves perfect groundedness and relevance for 2 of 3 questions, while the remaining response still scores strongly (0.8 groundedness, 0.9 relevance), showing improved evidence alignment and factual support.

## Actionable Insights and Business Recommendations

### Actionable Insights

1. **RAG improves clinical search:** Semantic retrieval helps overcome the limitations of keyword-based search by understanding clinically phrased queries.
2. **Groundedness remains important:** Relevant retrieval does not always guarantee that the final answer is fully supported by the source material.

3. **RAG improves grounding:** Retrieved context helps keep responses aligned with the available medical information.

4. **Evaluation builds trust:** Measuring relevance and groundedness helps ensure that responses remain useful and evidence-based.



### Recommendations

1. **Adopt the RAG model:** Use RAG as the default configuration to maintain grounded and relevant responses.
2. **Use Tuned RAG:** Tune retrieval and generation parameters to further improve response quality and grounding.
3. **Provide source citations:** Show the supporting medical reference with each response so physicians can verify the information.
4. **Expand validation:** Test the tuned RAG configuration on the planned 50 physician-authored queries before wider adoption.
5. **Introduce an evaluation gate:** Flag or reject responses with groundedness below the **0.90** target before presenting them as clinical reference answers.


<font size=6 color='blue'>Power Ahead</font>
___

## [OPTIONAL] Tuned RAG

Now, we evaluate each RAG configuration using the same set of questions and compare their groundedness and relevance scores.

- **Config 1 - Precision Focused:** Uses fewer retrieved chunks and lower randomness to prioritize concise, grounded responses.
- **Config 2 - Balanced:** Uses moderate retrieval and generation settings to balance context, grounding, and response length.
- **Config 3 - Context Heavy:** Retrieves more chunks and allows longer responses for questions requiring broader context.

In [51]:
RAG_CONFIGURATIONS = [

    {
        "name": "Config_1_Precision_Focused",
        "k": 3,
        "temperature": 0.0,
        "top_p": 0.9,
        "max_tokens": 400,
        "rationale": (
            "Retrieves fewer, highly relevant chunks with low randomness. "
            "Suitable for concise clinical reference answers."
        ),
    },

    {
        "name": "Config_2_Balanced",
        "k": 4,
        "temperature": 0.0,
        "top_p": 0.95,
        "max_tokens": 600,
        "rationale": (
            "Balances evidence coverage, groundedness, and response length. "
            "Suitable for most clinical reference queries."
        ),
    },

    {
        "name": "Config_3_Context_Heavy",
        "k": 6,
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 800,
        "rationale": (
            "Retrieves broader clinical context for complex or multi-part queries. "
            "Useful when answers require evidence from multiple sections."
        ),
    },

]

print("Defined RAG configurations:\n")

for config in RAG_CONFIGURATIONS:
    print(f"{config['name']}")
    print(f"Rationale: {config['rationale']}\n")

Defined RAG configurations:

Config_1_Precision_Focused
Rationale: Retrieves fewer, highly relevant chunks with low randomness. Suitable for concise clinical reference answers.

Config_2_Balanced
Rationale: Balances evidence coverage, groundedness, and response length. Suitable for most clinical reference queries.

Config_3_Context_Heavy
Rationale: Retrieves broader clinical context for complex or multi-part queries. Useful when answers require evidence from multiple sections.



***Prompt***:

<font size=3 color="#4682B4"><b> Write a Python `evaluate_all_rag_configurations()` function that runs `generate_rag_response()` for every question and every configuration in `RAG_CONFIGURATIONS`, evaluates groundedness and relevance using `evaluate_answers()`, stores the configuration details, answer, context, and scores, and returns the results as a pandas DataFrame.

</font>

This function generates and evaluates responses for every question across all RAG configurations, then stores the results and evaluation scores in a DataFrame.

In [52]:
def evaluate_all_rag_configurations(questions):

    results = []

    for question_name, question in questions.items():

        for config in RAG_CONFIGURATIONS:

            answer, context_for_query = generate_rag_response(
                user_input=question,
                k=config["k"],
                temperature=config["temperature"],
                top_p=config["top_p"],
                max_tokens=config["max_tokens"]
            )

            groundedness, relevance = evaluate_answers(
                [question],
                [answer],
                [context_for_query]
            )

            results.append({
                "Question": question_name,
                "Configuration": config["name"],
                "K": config["k"],
                "Temperature": config["temperature"],
                "Top_P": config["top_p"],
                "Max_Tokens": config["max_tokens"],
                "Answer": answer,
                "Context": context_for_query,

                "Groundedness": groundedness[0].score,
                "Groundedness_Rationale": groundedness[0].rationale,

                "Relevance": relevance[0].score,
                "Relevance_Rationale": relevance[0].rationale
            })

    return pd.DataFrame(results)

In [53]:
tuned_rag_results_df = evaluate_all_rag_configurations(questions)

tuned_rag_results_df

,Question,Configuration,K,Temperature,Top_P,Max_Tokens,Answer,Context,Groundedness,Groundedness_Rationale,Relevance,Relevance_Rationale
0,Question_1,Config_1_Precision_Focused,3,0.0,0.90,400,Answer:\nThe common symptoms of appendicitis i...,[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer accurately reflects the information...,1.0,The answer directly addresses all important as...
1,Question_1,Config_2_Balanced,4,0.0,0.95,600,Answer:\nCommon symptoms of appendicitis inclu...,[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer accurately reflects the information...,1.0,The answer directly addresses all important as...
2,Question_1,Config_3_Context_Heavy,6,0.0,1.00,800,Answer:\nThe common symptoms of appendicitis i...,[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer accurately reflects the information...,1.0,The answer directly addresses the user's quest...
3,Question_2,Config_1_Precision_Focused,3,0.0,0.90,400,"Answer:\nSudden patchy hair loss, known as alo...",[Source: /content/medical_diagnosis_manual.pdf...,0.8,The answer is mostly grounded in the context p...,0.9,The answer directly addresses the question by ...
4,Question_2,Config_2_Balanced,4,0.0,0.95,600,"Answer:\nSudden patchy hair loss, known as alo...",[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer accurately reflects the information...,1.0,The answer directly addresses the user's quest...
5,Question_2,Config_3_Context_Heavy,6,0.0,1.00,800,"Answer:\nSudden patchy hair loss, known as alo...",[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer is fully grounded in the provided c...,1.0,The answer directly addresses the question by ...
6,Question_3,Config_1_Precision_Focused,3,0.0,0.90,400,Answer:\nInitial treatment for traumatic brain...,[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer is fully grounded in the provided c...,1.0,The answer directly addresses the question by ...
7,Question_3,Config_2_Balanced,4,0.0,0.95,600,Answer:\nInitial treatment for a person with a...,[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer is fully grounded in the provided c...,1.0,The answer directly addresses the question by ...
8,Question_3,Config_3_Context_Heavy,6,0.0,1.00,800,Answer:\nInitial treatment for traumatic brain...,[Source: /content/medical_diagnosis_manual.pdf...,1.0,The answer is fully grounded in the provided c...,1.0,The answer directly addresses the question by ...


In [54]:
tuned_rag_summary_df = (
    tuned_rag_results_df
    .groupby("Configuration")
    .agg(
        Groundedness=("Groundedness", "mean"),
        Relevance=("Relevance", "mean")
    )
    .reset_index()
)

In [55]:
tuned_rag_summary_df

,Configuration,Groundedness,Relevance
0,Config_1_Precision_Focused,0.933333,0.966667
1,Config_2_Balanced,1.000000,1.000000
2,Config_3_Context_Heavy,1.000000,1.000000


**Observation:**

- All three RAG configurations achieved a relevance score of **1.0**, while groundedness ranged from **0.93 to 1.0**.

- The Balanced and Context Heavy configurations achieved **1.0 groundedness**, while the Precision Focused configuration scored **0.93**.

#### Final Configuration Selection

In [56]:
baseline_summary = pd.DataFrame({
    "Configuration": ["Baseline RAG"],
    "Groundedness": [result_df["rag_groundedness"].mean()],
    "Relevance": [result_df["rag_relevance"].mean()]
})

comparison_df = pd.concat(
    [baseline_summary, tuned_rag_summary_df],
    ignore_index=True
)

In [57]:
comparison_df

,Configuration,Groundedness,Relevance
0,Baseline RAG,0.966667,1.000000
1,Config_1_Precision_Focused,0.933333,0.966667
2,Config_2_Balanced,1.000000,1.000000
3,Config_3_Context_Heavy,1.000000,1.000000


**Observations**

- Tuning improves RAG performance, with groundedness and relevance increasing from **0.93/0.97** in the baseline to **1.0/1.0** in the tuned configurations.
